In [1]:
"""
GIS Property Intelligence — Address-Driven Spatial RAG
Supported input formats: FileGDB, Shapefile, CSV, GeoJSON, JSON
"""

import json
import math
import os
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import chromadb
from pydantic import BaseModel, Field, field_validator
from typing import Any
from sklearn.neighbors import BallTree

# ── 1. Configuration ──────────────────────────────────────────────────────────
OLLAMA_HOSTS = [
    "http://10.10.10.100:11434",
    "https://ollama.splsystems.in",
]

def resolve_ollama_host(hosts: list[str]) -> str:
    for host in hosts:
        try:
            resp = requests.get(host, timeout=5)
            if resp.status_code < 500:
                return host
        except requests.exceptions.RequestException:
            continue
    return hosts[-1]

OLLAMA_HOST = resolve_ollama_host(OLLAMA_HOSTS)
EMBED_MODEL = "nomic-embed-text-v2-moe:latest"
LLM_MODEL   = "gemma4:latest"
MAX_RECORDS = 5000
TOP_K       = 10   # BallTree spatial neighbours
CHROMA_TOP_K = 5   # ChromaDB semantic documents
CHROMA_PATH  = "./chroma_gis"
COLLECTION_NAME = "address_points"
FILE_PATH    = "Address_Points.geojson" # Target dataset

# Column aliases for universal loading
COLUMN_ALIASES: dict[str, list[str]] = {
    "address":   ["address", "addr", "full_address", "site_address"],
    "city":      ["city", "municipality"],
    "state":     ["state", "st"],
    "zipcode":   ["zipcode", "zip", "postal_code"],
    "latitude":  ["latitude", "lat", "y", "POINT_Y"],
    "longitude": ["longitude", "lon", "lng", "x", "POINT_X"],
}

EARTH_RADIUS_MILES = 3_958.8

# Global state variables (initialized during setup)
validated = []
tree = None
collection = None

# ── 2. Pre-Flight Diagnostics ─────────────────────────────────────────────────
def run_pre_flight_checks():
    print("[DEBUG] Running Pre-Flight Infrastructure Checks...")
    
    def check_endpoint(url: str, name: str):
        try:
            resp = requests.get(url, timeout=5)
            print(f"[DEBUG] [OK] {name} is reachable (Status: {resp.status_code})")
        except requests.exceptions.RequestException as e:
            print(f"[DEBUG] [FAIL] {name} is unreachable at {url}. Error: {e}")

    check_endpoint(OLLAMA_HOST, "Ollama Server")
    check_endpoint("https://nominatim.openstreetmap.org/status.php?format=json", "Nominatim Geocoder")
    print("-" * 50)

# ── 3. Data Loading & Normalization ───────────────────────────────────────────
def load_and_normalize(filepath: str) -> list[dict]:
    print(f"[DEBUG] Attempting to load dataset from: {filepath}")
    
    if not os.path.exists(filepath):
        print(f"[DEBUG] [WARNING] File not found: {filepath}. Creating 10 dummy records for testing pipeline.")
        return [{
            "address": f"{100+i} Test Ave", "city": "Testville", "state": "TX", 
            "zipcode": "75000", "latitude": 32.7 + (i*0.01), "longitude": -96.8 + (i*0.01),
            "zoning": "R1" if i % 2 == 0 else "C2"
        } for i in range(10)]

    try:
        if filepath.endswith(('.geojson', '.shp', '.gdb')):
            df = gpd.read_file(filepath)

            if MAX_RECORDS:
                df = df.head(MAX_RECORDS)

            print(f"[DEBUG] Processing {len(df):,} records")
            if df.geometry is not None and 'latitude' not in df.columns:
                df["latitude"] = df.geometry.y
                df["longitude"] = df.geometry.x
        else:
            df = pd.read_csv(filepath)
            
        print(f"[DEBUG] Successfully loaded {len(df)} rows.")
        
        normalized_records = []
        for _, row in df.iterrows():
            rec = row.to_dict()

            # Construct base properties from known explicit headers
            norm_rec = {
                "address": (
                    f"{rec.get('ADDRNMBR','')} "
                    f"{rec.get('ADDRNAME','')} "
                    f"{rec.get('ADDRSFX','')}"
                ).strip(),
                "city": rec.get("COMMUNITY"),
                "state": rec.get("STATE"),
                "zipcode": str(rec.get("ADDRZIP") if pd.notna(rec.get("ADDRZIP")) else ""),
                "latitude": rec.get("latitude"),
                "longitude": rec.get("longitude"),
                "attributes": {}
            }

            # List of keys we already handled manually above
            explicit_keys = [
                "ADDRNMBR", "ADDRNAME", "ADDRSFX", "COMMUNITY", 
                "STATE", "ADDRZIP", "latitude", "longitude", "geometry"
            ]

            # Map the remaining attributes and check aliases
            for key, val in rec.items():
                if key in explicit_keys:
                    continue
                
                matched = False
                for canonical, aliases in COLUMN_ALIASES.items():
                    if key.lower() in [a.lower() for a in aliases]:
                        if not norm_rec.get(canonical): # Prevent overwriting if already populated
                            norm_rec[canonical] = val
                        matched = True
                        break
                        
                if not matched and key != 'geometry':
                    norm_rec["attributes"][key] = val
                    
            normalized_records.append(norm_rec)
            
        print(f"[DEBUG] Successfully normalized {len(normalized_records)} records.")
        return normalized_records
    except Exception as e:
        print(f"[DEBUG] [FATAL] Data loading failed: {e}")
        return []

# ── 4. Pydantic Models ────────────────────────────────────────────────────────
class AddressPoint(BaseModel):
    address:    str
    city:       str
    state:      str
    zipcode:    str
    latitude:   float
    longitude:  float
    attributes: dict[str, Any] = Field(default_factory=dict)

    @field_validator("address", "city", "state", "zipcode", mode="before")
    @classmethod
    def coerce_str(cls, v: Any) -> str:
        return str(v) if pd.notna(v) and v is not None else ""

    @field_validator("latitude", "longitude", mode="before")
    @classmethod
    def coerce_float(cls, v: Any) -> float:
        try:
            f = float(v)
            return f if math.isfinite(f) else 0.0
        except (TypeError, ValueError):
            return 0.0

def validate_records(records: list[dict]) -> tuple[list[AddressPoint], list[dict]]:
    print(f"[DEBUG] Starting Pydantic validation for {len(records)} records...")
    valid, failed = [], []
    for i, rec in enumerate(records):
        try:
            point = AddressPoint(**rec)
            if point.latitude == 0.0 and point.longitude == 0.0:
                failed.append({"index": i, "reason": "zero lat/lon", "record": rec})
            else:
                valid.append(point)
        except Exception as exc:
            failed.append({"index": i, "reason": str(exc), "record": rec})
            
    print(f"[DEBUG] Validation Complete. Valid: {len(valid)}, Failed: {len(failed)}")
    return valid, failed

# ── 5. Spatial Indexing & Geocoding ───────────────────────────────────────────
def build_spatial_index(valid_records: list[AddressPoint]):
    global tree
    if not valid_records:
        print("[DEBUG] [WARNING] No valid records. Skipping BallTree build.")
        return
        
    coords = np.array(
        [[math.radians(p.latitude), math.radians(p.longitude)] for p in valid_records],
        dtype=np.float64,
    )
    tree = BallTree(coords, metric="haversine")
    print(f"[DEBUG] BallTree spatial index built successfully. Indexed {len(coords)} coordinates.")

def spatial_query(lat: float, lon: float, k: int = TOP_K) -> list[dict]:
    if not tree or not validated:
        return []
    query = np.array([[math.radians(lat), math.radians(lon)]])
    distances, indices = tree.query(query, k=min(k, len(validated)))

    results = []
    for dist_rad, idx in zip(distances[0], indices[0]):
        rec = validated[idx].model_dump()
        rec["distance_miles"] = round(dist_rad * EARTH_RADIUS_MILES, 4)
        results.append(rec)
    return results

def geocode(address: str) -> tuple[float, float] | None:
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "GIS-Property-Intelligence/1.0"}
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=10)
        resp.raise_for_status()
        hits = resp.json()
        if hits:
            return float(hits[0]["lat"]), float(hits[0]["lon"])
        print(f"[DEBUG] Geocode returned no results for: {address}")
    except Exception as exc:
        print(f"[DEBUG] [ERROR] Geocode failed: {exc}")
    return None

# ── 6. Embeddings & ChromaDB ──────────────────────────────────────────────────
def build_geo_document(point: dict, rank: int) -> str:
    attrs = point.get("attributes", {})
    return (
        f"Address: {point.get('address','')}, {point.get('city','')} {point.get('zipcode','')}\n"
        f"Distance: {point.get('distance_miles', 'N/A')} miles\n"
        f"Zoning: {attrs.get('zoning', 'Unknown')}\n"
    ).strip()

def embed(texts: list[str]) -> list[list[float]]:
    url = f"{OLLAMA_HOST}/api/embed"
    body = {"model": EMBED_MODEL, "input": texts}
    try:
        resp = requests.post(url, json=body, timeout=120)
        resp.raise_for_status()
        return resp.json()["embeddings"]
    except Exception as e:
        print(f"[DEBUG] [ERROR] Embedding API failure: {e}")
        return [[0.0]*768 for _ in texts]

def build_vector_db(valid_records: list[AddressPoint]):
    global collection
    print("[DEBUG] Initializing ChromaDB...")
    chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    collection = chroma_client.create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

    BATCH_SIZE = 64
    print(f"[DEBUG] Starting ChromaDB Vector Insertion (Batch Size: {BATCH_SIZE})...")

    for batch_start in range(0, len(valid_records), BATCH_SIZE):
        batch = valid_records[batch_start : batch_start + BATCH_SIZE]
        docs = [build_geo_document(p.model_dump(), i) for i, p in enumerate(batch)]
        ids = [f"addr_{batch_start + i}" for i in range(len(batch))]
        metas = [{"address": p.address, "latitude": p.latitude, "longitude": p.longitude} for p in batch]
        
        vectors = embed(docs)
        if vectors and any(v for v in vectors):
            collection.add(ids=ids, documents=docs, embeddings=vectors, metadatas=metas)
        
        print(f"[DEBUG] Inserted batch {batch_start} to {batch_start + len(batch)}")

    print(f"[DEBUG] ChromaDB populated. Total docs: {collection.count()}")

# ── 7. RAG Controller ─────────────────────────────────────────────────────────
def ask_address(address: str, question: str, verbose: bool = False) -> str:
    print(f"\n[DEBUG] --- Starting RAG Pipeline for query: '{question}' ---")
    
    coords = geocode(address)
    if coords is None:
        return f"[ERROR] Could not geocode: '{address}'"
    lat, lon = coords
    if verbose: print(f"[DEBUG] Geocoded {address} -> Lat: {lat}, Lon: {lon}")

    nearby = spatial_query(lat, lon, k=TOP_K)
    if verbose: print(f"[DEBUG] Found {len(nearby)} nearby features via BallTree.")

    context = {"coordinates": {"lat": lat, "lon": lon}, "nearest": nearby[0] if nearby else None}
    
    q_vec = embed([question])[0]
    results = collection.query(query_embeddings=[q_vec], n_results=CHROMA_TOP_K)
    retrieved = results["documents"][0] if results and results.get("documents") else []
    
    if verbose: print(f"[DEBUG] Retrieved {len(retrieved)} semantic documents from ChromaDB.")

    prompt = f"""
You are a GIS Address Intelligence Assistant.

Target Address:
{address}

Coordinates:
{lat}, {lon}

Question:
{question}

Nearby Address Points:
{json.dumps(nearby, indent=2)}

Retrieved Documents:
{json.dumps(retrieved, indent=2)}

Rules:

1. Use ALL nearby address points.
2. Mention distances.
3. Mention communities.
4. Mention ZIP codes.
5. Mention parcel identifiers when available.
6. Do not invent information.
7. If information is unavailable, explicitly say so.

Answer:
"""
    
    if verbose: print("[DEBUG] Sending prompt to Gemma4...")
    url = f"{OLLAMA_HOST}/api/generate"
    body = {"model": LLM_MODEL, "prompt": prompt, "stream": False}
    
    try:
        resp = requests.post(url, json=body, timeout=120)
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except Exception as e:
        return f"[DEBUG] [ERROR] LLM Generation failed: {e}"

# ── 8. Main Execution ─────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=== Initializing GIS RAG System ===")
    
    # 1. Check endpoints
    run_pre_flight_checks()
    
    # 2. Load and validate data
    raw_records = load_and_normalize(FILE_PATH)
    validated, failed_records = validate_records(raw_records)
    
    print("=" * 80)
    print("VALIDATION DEBUG")
    print("=" * 80)

    print("Validated:", len(validated))
    print("Failed:", len(failed_records))

    if failed_records:
        print("\nFirst Failed Record:\n")
        print(json.dumps(failed_records[0], indent=2, default=str))
    
    # 3. Build spatial and vector indices
    build_spatial_index(validated)
    build_vector_db(validated)
    
    print("\n=== System Ready. Running Test Queries ===")
    
# Example 1
ans1 = ask_address(
    "949 Sapphire St",
    "What are the closest address points to this location?",
    verbose=True
)
print(f"\nResponse 1:\n{ans1}\n")

# Example 2
ans2 = ask_address(
    "949 Sapphire St",
    "Which community, ZIP code, and parcel information are nearby?",
    verbose=True
)
print(f"\nResponse 2:\n{ans2}\n")

=== Initializing GIS RAG System ===
[DEBUG] Running Pre-Flight Infrastructure Checks...


KeyboardInterrupt: 